In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-mpf qiskit-addon-utils scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# 트로터 오류를 줄이기 위한 다중 곱 공식

import TutorialFeedback from '@site/src/components/TutorialFeedback';






*예상 QPU 사용 시간: Heron r2 프로세서에서 약 4분 (참고: 이는 추정치이며 실제 실행 시간은 다를 수 있습니다.)*

## 배경
이 튜토리얼에서는 다중 곱 공식(MPF, Multi-Product Formula)을 사용하여 실제로 실행할 가장 깊은 트로터 Circuit으로 인한 오류보다 관측값에 대한 트로터 오류를 낮추는 방법을 설명합니다.
MPF는 여러 Circuit 실행의 가중 조합을 통해 해밀토니안 동역학의 트로터 오류를 줄입니다. 해밀토니안 $H$에 대해 양자 상태 $\rho(t)=e^{-i H t} \rho(0) e^{i H t}$의 관측값 기댓값을 구하는 작업을 생각해 보겠습니다. 곱 공식(PF, Product Formula)을 사용하면 다음과 같이 시간 발전 $e^{-i H t}$를 근사할 수 있습니다:

- 해밀토니안 $H$를 $H=\sum_{a=1}^d F_a$로 작성합니다. 여기서 $F_a$는 에르미트 연산자로, 각 대응 유니터리가 양자 디바이스에서 효율적으로 구현될 수 있습니다.
- 서로 가환하지 않는 항 $F_a$를 근사합니다.

그러면 1차 PF(Lie-Trotter 공식)는 다음과 같습니다:

$$S_1(t):=\prod_{a=1}^d e^{-i F_a t},$$

이는 2차 오류 항 $S_1(t)=e^{-i H t}+\mathcal{O}\left(t^{2}\right)$을 가집니다. 더 빠르게 수렴하는 고차 PF(Lie-Trotter-Suzuki 공식)도 사용할 수 있으며, 이는 다음과 같이 재귀적으로 정의됩니다:

$$S_2(t):=\prod_{a=1}^d e^{-i F_a t/2}\prod_{a=1}^d e^{-i F_a t/2}$$

$$S_{2 \chi}(t):= S_{2 \chi -2}(s_{\chi}t)^2 S_{2 \chi -2}((1-4s_{\chi})t)S_{2 \chi -2}(s_{\chi}t)^2,$$

여기서 $\chi$는 대칭 PF의 차수이고 $s_p = \left( 4 - 4^{1/(2p-1)} \right)^{-1}$입니다. 장시간 발전의 경우 시간 구간 $t$를 $t/k$ 길이의 $k$개 구간(트로터 스텝)으로 나누고, 각 구간에서 $\chi$차 곱 공식 $S_{\chi}$로 시간 발전을 근사할 수 있습니다. 따라서 $k$개의 트로터 스텝에 걸친 시간 발전 연산자에 대한 $\chi$차 PF는 다음과 같습니다:

$$ S_{\chi}^{k}(t) = \left[ S_{\chi} \left( \frac{t}{k} \right)\right]^k = e^{-i H t}+O\left(t \left( \frac{t}{k} \right)^{\chi} \right)$$

여기서 오류 항은 트로터 스텝 수 $k$와 PF의 차수 $\chi$가 증가할수록 감소합니다.

정수 $k \geq 1$과 곱 공식 $S_{\chi}(t)$가 주어지면, 근사 시간 발전 상태 $\rho_k(t)$는 $\rho_0$에 곱 공식 $S_{\chi}\left(\frac{t}{k}\right)$를 $k$번 적용하여 얻을 수 있습니다.

$$
\rho_k(t)=S_{\chi}\left(\frac{t}{k}\right)^k \rho_0 S_{\chi}\left(\frac{t}{k}\right)^{-k}
$$

$\rho_k(t)$는 트로터 근사 오류 ||$\rho_k(t)-\rho(t) ||$를 가지는 $\rho(t)$의 근사값입니다. $\rho(t)$의 트로터 근사들의 선형 조합을 고려하면:

$$
\mu(t) = \sum_{j}^{l} x_j \rho^{k_j}_{j}\left(\frac{t}{k_j}\right) + \text{some remaining Trotter error} \, ,
$$

여기서 $x_j$는 가중 계수, $\rho^{k_j}_j$는 곱 공식 $S^{k_j}_{\chi}$로 초기 상태를 $k_j$개의 트로터 스텝만큼 발전시켜 얻은 순수 상태에 해당하는 밀도 행렬이며, $j \in {1, ..., l}$은 MPF를 구성하는 PF의 수에 대한 인덱스입니다. $\mu(t)$의 모든 항은 동일한 곱 공식 $S_{\chi}(t)$를 기반으로 합니다.
목표는 $\|\mu(t)-\rho(t)\|$가 더 낮은 $\mu(t)$를 찾아 ||$\rho_k(t)-\rho(t) \|$를 개선하는 것입니다.

* $x_i$가 양수일 필요가 없으므로 $\mu(t)$는 물리적 상태일 필요가 없습니다. 여기서의 목표는 관측값의 기댓값 오류를 최소화하는 것이며, $\rho(t)$의 물리적 대체물을 찾는 것이 아닙니다.
* $k_j$는 Circuit 깊이와 트로터 근사 수준 모두를 결정합니다. $k_j$ 값이 작을수록 Circuit이 짧아져 Circuit 오류는 줄지만 원하는 상태에 대한 근사는 덜 정확해집니다.

핵심은 $\mu(t)$로 주어지는 나머지 트로터 오류가 가장 큰 $k_j$ 값을 단순히 사용했을 때 얻는 트로터 오류보다 작다는 점입니다.

이 방법의 유용성은 두 가지 관점에서 볼 수 있습니다:

1. 실행 가능한 고정된 트로터 스텝 예산에 대해, 총 트로터 오류가 더 작은 결과를 얻을 수 있습니다.
2. 실행하기에 너무 큰 목표 트로터 스텝 수가 주어졌을 때, MPF를 사용하여 유사한 트로터 오류를 달성하는 더 낮은 깊이의 Circuit 컬렉션을 실행할 수 있습니다.
## 요구 사항
이 튜토리얼을 시작하기 전에 다음 패키지가 설치되어 있는지 확인하세요:

* Qiskit SDK v1.0 이상 ([시각화](https://docs.quantum.ibm.com/api/qiskit/visualization) 지원 포함)
* Qiskit Runtime v0.22 이상 (`pip install qiskit-ibm-runtime`)
* MPF Qiskit 애드온 (`pip install qiskit_addon_mpf`)
* Qiskit 애드온 유틸리티 (`pip install qiskit_addon_utils`)
* Quimb 라이브러리 (`pip install quimb`)
* Qiskit Quimb 라이브러리 (`pip install qiskit-quimb`)
* 패키지 간 호환성을 위한 Numpy v0.21 (`pip install numpy==0.21`)
## 파트 I. 소규모 예제
### MPF의 안정성 탐색
MPF 상태 $\mu(t)$를 구성하는 트로터 스텝 수 $k_j$의 선택에는 명백한 제한이 없습니다. 하지만 $\mu(t)$로부터 계산되는 기댓값의 불안정성을 피하기 위해 신중하게 선택해야 합니다. 일반적으로 좋은 규칙은 최소 트로터 스텝 $k_{\text{min}}$을 $t/k_{\text{min}} \lt 1$이 되도록 설정하는 것입니다. 이에 대한 자세한 내용과 다른 $k_j$ 값 선택 방법은 [MPF에 대한 트로터 스텝 선택 방법](https://qiskit.github.io/qiskit-addon-mpf/how_tos/choose_trotter_steps.html) 가이드를 참고하세요.

아래 예제에서는 다양한 시간 범위에 걸쳐 서로 다른 시간 발전 상태를 사용하여 자화의 기댓값을 계산함으로써 MPF 솔루션의 안정성을 탐색합니다. 구체적으로, 각 트로터 스텝으로 구현된 근사 시간 발전에서 계산된 기댓값과 다양한 MPF 모델(정적 및 동적 계수)의 기댓값을 시간 발전된 관측값의 정확한 값과 비교합니다. 먼저 트로터 공식의 매개변수와 발전 시간을 정의해 보겠습니다.

In [1]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
from functools import partial
from copy import deepcopy

from qiskit import QuantumCircuit
from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector
from qiskit.synthesis import SuzukiTrotter
from qiskit.transpiler import CouplingMap, PassManager
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit.library import XXPlusYYGate
from qiskit.transpiler.passes.optimization.collect_and_collapse import (
    CollectAndCollapse,
    collect_using_filter_function,
    collapse_to_operation,
)

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

from qiskit_addon_utils.problem_generators import (
    generate_xyz_hamiltonian,
    generate_time_evolution_circuit,
)
from qiskit_addon_utils.slicing import slice_by_depth
from qiskit_addon_mpf.static import setup_static_lse
from qiskit_addon_mpf.dynamic import setup_dynamic_lse
from qiskit_addon_mpf.costs import (
    setup_exact_problem,
    setup_sum_of_squares_problem,
    setup_frobenius_problem,
)
from qiskit_addon_mpf.backends.tenpy_layers import (
    LayerModel,
    LayerwiseEvolver,
)
from qiskit_addon_mpf.backends.tenpy_tebd import MPOState, MPS_neel_state

from scipy.linalg import expm

# Suppress TeNPy's `unit_cell_width` future-API warning. The default
# (`unit_cell_width=len(sites)`) is correct for Chain lattices, which is what
# `CouplingMap.from_line(...)` produces here, so the warning is informational.
warnings.filterwarnings(
    "ignore",
    message=r".*unit_cell_width.*",
    category=UserWarning,
)


# --- Helper: collect XX + YY rotations into a single gate ---
def filter_function(node):
    return node.op.name in {"rxx", "ryy"}


collect_function = partial(
    collect_using_filter_function,
    filter_function=filter_function,
    split_blocks=True,
    min_block_size=1,
)


def collapse_to_xx_plus_yy(block):
    param = 0.0
    for node in block.data:
        param += node.operation.params[0]
    return XXPlusYYGate(param)


collapse_function = partial(
    collapse_to_operation,
    collapse_function=collapse_to_xx_plus_yy,
)

pm = PassManager()
pm.append(CollectAndCollapse(collect_function, collapse_function))

이 예제에서는 초기 상태로 Neel 상태 $\vert \text{Neel} \rangle = \vert 0101...01 \rangle$를 사용하고, 10개 사이트로 이루어진 선형 Heisenberg 모델을 시간 발전을 지배하는 해밀토니안으로 사용합니다.

$$
\hat{\mathcal{H}}_{Heis} = J \sum_{i=1}^{L-1} \left(X_i X_{(i+1)}+Y_i Y_{(i+1)}+ Z_i Z_{(i+1)} \right) \, ,
$$

여기서 $J$는 최근접 이웃 간의 결합 강도입니다.

In [2]:
L = 10

# Generate coupling map and Hamiltonian
coupling_map = CouplingMap.from_line(L, bidirectional=False)

hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(1.0, 1.0, 1.0),
    ext_magnetic_field=(0.0, 0.0, 0.0),
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIXXI', 'IIIIIIIYYI', 'IIIIIIIZZI', 'IIIIIXXIII', 'IIIIIYYIII', 'IIIIIZZIII', 'IIIXXIIIII', 'IIIYYIIIII', 'IIIZZIIIII', 'IXXIIIIIII', 'IYYIIIIIII', 'IZZIIIIIII', 'IIIIIIIIXX', 'IIIIIIIIYY', 'IIIIIIIIZZ', 'IIIIIIXXII', 'IIIIIIYYII', 'IIIIIIZZII', 'IIIIXXIIII', 'IIIIYYIIII', 'IIIIZZIIII', 'IIXXIIIIII', 'IIYYIIIIII', 'IIZZIIIIII', 'XXIIIIIIII', 'YYIIIIIIII', 'ZZIIIIIIII'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])


In [3]:
# Observable: ZZ on the middle pair of qubits
observable = SparsePauliOp.from_sparse_list(
    [("ZZ", (L // 2 - 1, L // 2), 1.0)], num_qubits=L
)
print(observable)

SparsePauliOp(['IIIIZZIIII'],
              coeffs=[1.+0.j])


In [4]:
# MPF parameters
mpf_trotter_steps = [1, 2, 4]
order = 2
symmetric = False

trotter_times = np.arange(0.5, 1.55, 0.1)
exact_evolution_times = np.arange(trotter_times[0], 1.55, 0.05)

#### Build Trotter circuits

We create the circuits implementing the approximate Trotter time-evolutions for each time point and each Trotter step count. The `CollectAndCollapse` pass defined in the Setup section collects XX and YY rotations into single XX+YY gates, to prepare for more efficient tensor-network simulation later.

In [5]:
# Initial Neel state preparation
initial_state_circ = QuantumCircuit(L)
initial_state_circ.x([i for i in range(L) if i % 2 != 0])


all_circs = []
for total_time in trotter_times:
    mpf_trotter_circs = [
        generate_time_evolution_circuit(
            hamiltonian,
            time=total_time,
            synthesis=SuzukiTrotter(reps=num_steps, order=order),
        )
        for num_steps in mpf_trotter_steps
    ]

    mpf_trotter_circs = pm.run(
        mpf_trotter_circs
    )  # Collect XX and YY into XX + YY

    mpf_circuits = [
        initial_state_circ.compose(circuit) for circuit in mpf_trotter_circs
    ]
    all_circs.append(mpf_circuits)

In [6]:
mpf_circuits[-1].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/multi-product-formula/extracted-outputs/c7ee61e7-0.avif" alt="Output of the previous code cell" />

그런 다음 근사 트로터 시간 발전을 구현하는 Circuit을 생성합니다.

In [7]:
aer_sim = AerSimulator()
pm_sim = generate_preset_pass_manager(backend=aer_sim, optimization_level=3)

isa_circs_all_times = [
    pm_sim.run([deepcopy(c) for c in mpf_circuits])
    for mpf_circuits in all_circs
]

### Step 3: Execute using Qiskit primitives

For the small-scale example we run the ISA-lowered Trotter circuits through the `EstimatorV2` primitive backed by Aer. Doing so gives us a *noiseless* reference value for each $(k_j, t)$ pair &mdash; these are the $\langle A \rangle_{k_j}(t)$ values that the MPF will combine in Step 4. We sweep over evolution times so that we can later plot the full time-series curve of each individual product formula and of the MPF.

In [8]:
estimator = Estimator(mode=aer_sim)

mpf_expvals_all_times, mpf_stds_all_times = [], []
for isa_circuits in isa_circs_all_times:
    result = estimator.run(
        [(circuit, observable) for circuit in isa_circuits], precision=0.005
    ).result()
    mpf_expvals_all_times.append([res.data.evs for res in result])
    mpf_stds_all_times.append([res.data.stds for res in result])

![Output of the previous code cell](../docs/images/tutorials/multi-product-formula/extracted-outputs/92dc20a7-0.avif)

다음으로 트로터 Circuit에서 시간 발전된 기댓값을 계산합니다.

In [9]:
exact_expvals = []
for t in exact_evolution_times:
    exp_H = expm(-1j * t * hamiltonian.to_matrix())
    initial_state = Statevector(initial_state_circ).data
    time_evolved_state = exp_H @ initial_state

    exact_obs = (
        time_evolved_state.conj()
        @ observable.to_matrix()
        @ time_evolved_state
    ).real
    exact_expvals.append(exact_obs)

비교를 위해 정확한 기댓값도 계산합니다.

In [10]:
lse = setup_static_lse(mpf_trotter_steps, order=order, symmetric=symmetric)

#### 정적 MPF 계수
정적 MPF는 $x_j$ 값이 진화 시간 $t$에 의존하지 않는 경우입니다. 차수 $\chi = 1$인 PF에서 $k_j$ 트로터 스텝을 사용하는 경우를 고려하면 다음과 같이 쓸 수 있습니다:

$$ S_1^{k_j}\left( \frac{t}{k_j} \right)=e^{-i H t}+ \sum_{n=1}^{\infty} A_n \frac{t^{n+1}}{k_j^n}  $$

여기서 $A_n$은 해밀토니안 분해에서 $F_a$ 항들의 교환자(commutator)에 의존하는 행렬입니다. $A_n$ 자체는 시간과 트로터 스텝 수 $k_j$에 독립적이라는 점이 중요합니다. 따라서 가중치 $x_j$를 신중하게 선택하면 $\mu(t)$에 기여하는 낮은 차수의 오류 항을 상쇄할 수 있습니다. $\mu(t)$ 표현식에서 처음 $l-1$개 항(트로터 스텝 수가 적어 가장 큰 기여를 하는 항)의 트로터 오류를 상쇄하려면, 계수 $x_j$가 다음 방정식을 만족해야 합니다:

$$ \sum_{j=1}^l x_j = 1 $$
$$ \sum_{j=1}^{l-1} \frac{x_j}{k_j^{n}} = 0 $$

여기서 $n=0, ... l-2$입니다. 첫 번째 방정식은 구성된 상태 $\mu(t)$에 편향이 없음을 보장하며, 두 번째 방정식은 트로터 오류의 상쇄를 보장합니다. 고차 PF의 경우, 두 번째 방정식은 $ \sum_{j=1}^{l-1} \frac{x_j}{k_j^{\eta}} = 0 $이 되며, 여기서 $\eta = \chi + 2n$ (대칭 PF의 경우) 또는 $\eta = \chi + n$ (그 외의 경우), $n=0, ..., l-2$입니다. 결과적인 오류 (참고문헌 [\[1\]](#references), [\[2\]](#references))는 다음과 같습니다:

$$ \epsilon = \mathcal{O} \left( \frac{t^{l+1}}{k_1^l} \right).$$

주어진 $k_j$ 값 집합에 대한 정적 MPF 계수를 결정하는 것은, 변수 $x_j$에 대해 위 두 방정식으로 정의된 선형 연립방정식 $Ax=b$를 푸는 것과 같습니다. 여기서 $x$는 구하고자 하는 계수, $A$는 $k_j$와 사용하는 PF 유형($S$)에 의존하는 행렬, $b$는 제약 조건 벡터입니다. 구체적으로:

$$A_{0,j} = 1 $$
$$A_{i>0,j} = k_{j}^{-(\chi + s(i-1))}$$
$$b_0 = 1$$
$$b_{i>0} = 0 $$

여기서 $\chi$는 ``order``(차수), $s$는 ``symmetric``이 ``True``이면 $2$, 아니면 $1$, $k_{j}$는 ``trotter_steps``, $x$는 풀어야 할 변수입니다. 인덱스 $i$와 $j$는 $0$에서 시작합니다. 이를 행렬 형태로도 시각화할 수 있습니다:

$$
A =
\begin{bmatrix}
A_{0,0} & A_{0,1} & A_{0,2} & ... \\
A_{1,0} & A_{1,1} & A_{1,2} & ...  \\
A_{2,0} & A_{2,1} & A_{2,2} & ...  \\
... & ... & ... & ...
\end{bmatrix} =
\begin{bmatrix}
1 & 1 & 1 & ... \\
k_{0}^{-(\chi + s(1-1))} & k_{1}^{-(\chi + s(1-1))} & k_{2}^{-(\chi + s(1-1))} & ... \\
k_{0}^{-(\chi + s(2-1))} & k_{1}^{-(\chi + s(2-1))} & k_{2}^{-(\chi + s(2-1))} & ... \\
... & ... & ... & ...
\end{bmatrix}
$$

그리고

$$
b =
\begin{bmatrix}
b_{0} \\
b_{1} \\
b_{2}  \\
...
\end{bmatrix} =
\begin{bmatrix}
1 \\
0 \\
0  \\
...
\end{bmatrix}
$$

자세한 내용은 선형 연립방정식([LSE](https://qiskit.github.io/qiskit-addon-mpf/stubs/qiskit_addon_mpf.static.LSE.html)) 문서를 참조하세요.

$x = A^{-1}b$로 분석적 해를 구할 수 있습니다. 예를 들어 참고문헌 [\[1\]](#references) 또는 [\[2\]](#references)를 참고하세요.
단, 이 정확한 해는 "불량 조건(ill-conditioned)"일 수 있어 계수 $x$의 L1-노름이 매우 커질 수 있으며, 이는 MPF의 성능 저하로 이어질 수 있습니다.
대신, L1-노름을 최소화하는 근사 해를 구하여 MPF 동작을 최적화하려 시도할 수도 있습니다.
##### LSE 설정
$k_j$ 값을 선택한 후에는 위에서 설명한 대로 먼저 LSE인 $Ax=b$를 구성해야 합니다.
행렬 $A$는 $k_j$뿐만 아니라 PF의 선택, 특히 그 _차수(order)_에도 의존합니다.
또한 `symmetric=True/False`를 설정하여 PF의 대칭 여부를 고려할 수 있습니다(참고문헌 [\[1\]](#references) 참조).
단, 참고문헌 [\[2\]](#references)에서 보여주듯이 이는 필수는 아닙니다.

In [11]:
lse.A

array([[1.      , 1.      , 1.      ],
       [1.      , 0.25    , 0.0625  ],
       [1.      , 0.125   , 0.015625]])

In [12]:
lse.b

array([1., 0., 0.])

With the LSE in hand, we solve for the static coefficients $x_j$ via `lse.solve()` (this is the direct $x = A^{-1}b$ solution).

In [13]:
mpf_coeffs = lse.solve()
print(
    f"The static coefficients associated with the ansatze are: {mpf_coeffs}"
)

The static coefficients associated with the ansatze are: [ 0.04761905 -0.57142857  1.52380952]


제약 조건 벡터 $b$의 원소는 다음과 같습니다:
$$ b_{0} = 1 $$
$$ b_1 = b_2 = 0 $$

따라서,

$$
b =
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

마찬가지로 `lse`에서도 확인할 수 있습니다:

In [14]:
model_exact, coeffs_exact = setup_exact_problem(lse)
model_exact.solve()
print(coeffs_exact.value)

[ 0.04761905 -0.57142857  1.52380952]


In [15]:
print(
    "L1 norm of the exact coefficients:",
    np.linalg.norm(coeffs_exact.value, ord=1),
)

L1 norm of the exact coefficients: 2.1428571428556378


`lse` 객체에는 연립방정식을 만족하는 정적 계수 $x_j$를 찾는 메서드가 있습니다.

In [16]:
model_approx, coeffs_approx = setup_sum_of_squares_problem(
    lse, max_l1_norm=1.5
)
model_approx.solve()
print(coeffs_approx.value)
print(
    "L1 norm of the approximate coefficients:",
    np.linalg.norm(coeffs_approx.value, ord=1),
)

[-1.10294118e-03 -2.48897059e-01  1.25000000e+00]
L1 norm of the approximate coefficients: 1.5


#### Dynamic MPF coefficients

The static MPF cancels Trotter-error terms in a Hamiltonian- and state-agnostic way, so it does not necessarily produce the smallest possible approximation error for a given Hamiltonian and initial state. The dynamic MPF (Refs. [\[2\]](#references), [\[3\]](#references)) instead finds time-dependent coefficients $x_i(t)$ that minimize the Frobenius-norm distance $\|\rho(t) - \mu^D(t)\|_F^2$ at each time $t$. As shown in the Background, this requires the overlap matrix $M_{ij}(t)$ between Trotter-evolved states and the overlap $L_i(t)$ with the exact state &mdash; both of which we estimate using tensor-network (TeNPy) backends in `qiskit_addon_mpf`.

To set up the dynamic LSE we need three ingredients:

1. An **approximate evolver factory** that the addon will run for each $k_j$ to produce $\rho_{k_j}(t)$ as an MPS/MPO. We build it from the layered structure of the order-$2$ Trotter circuit (one layer per `slice_by_depth`), wrapped as a `LayerwiseEvolver` with TeNPy truncation parameters.
2. An **exact evolver factory** that produces a high-accuracy reference $\rho(t)$. We use a small-time-step fourth-order Suzuki-Trotter circuit (`dt=0.1`, `order=4`) as a proxy for exact evolution.
3. An **identity factory** and an **initial-state MPS** that seed the TeNPy simulation.

The cell below constructs the approximate evolver factory.

In [17]:
# Create approximate time-evolution circuits
single_2nd_order_circ = generate_time_evolution_circuit(
    hamiltonian, time=1.0, synthesis=SuzukiTrotter(reps=1, order=order)
)
single_2nd_order_circ = pm.run(single_2nd_order_circ)  # collect XX and YY

# Find layers in the circuit
layers = slice_by_depth(single_2nd_order_circ, max_slice_depth=1)

# Create tensor network models
models = [
    LayerModel.from_quantum_circuit(layer, conserve="Sz") for layer in layers
]

# Create the time-evolution object
approx_factory = partial(
    LayerwiseEvolver,
    layers=models,
    options={
        "preserve_norm": False,
        "trunc_params": {
            "chi_max": 64,
            "svd_min": 1e-8,
            "trunc_cut": None,
        },
        "max_delta_t": 2,
    },
)

##### 정확한 모델을 이용한 $x$ 최적화
$x=A^{-1}b$를 계산하는 대신, [setup_exact_model](https://qiskit.github.io/qiskit-addon-mpf/stubs/qiskit_addon_mpf.static.setup_exact_model.html)을 사용하여 LSE를 제약 조건으로 사용하고 최적해가 $x$를 산출하는 [cvxpy.Problem](https://www.cvxpy.org/api_reference/cvxpy.problems.html#cvxpy.Problem) 인스턴스를 구성할 수도 있습니다.

다음 섹션에서 이 인터페이스가 왜 존재하는지 명확해질 것입니다.

In [18]:
single_4th_order_circ = generate_time_evolution_circuit(
    hamiltonian, time=1.0, synthesis=SuzukiTrotter(reps=1, order=4)
)
single_4th_order_circ = pm.run(single_4th_order_circ)
exact_model_layers = [
    LayerModel.from_quantum_circuit(layer, conserve="Sz")
    for layer in slice_by_depth(single_4th_order_circ, max_slice_depth=1)
]

exact_factory = partial(
    LayerwiseEvolver,
    layers=exact_model_layers,
    dt=0.1,
    options={
        "preserve_norm": False,
        "trunc_params": {
            "chi_max": 64,
            "svd_min": 1e-8,
            "trunc_cut": None,
        },
        "max_delta_t": 2,
    },
)

Finally, we define an `identity_factory` that yields the initial MPO state and prepare the Néel initial state as an MPS that matches the lattice used by the layered Trotter model.

In [19]:
def identity_factory():
    return MPOState.initialize_from_lattice(models[0].lat, conserve=True)


mps_initial_state = MPS_neel_state(models[0].lat)

이 계수들로 구성된 MPF가 좋은 결과를 낼지 여부의 지표로 L1-노름을 사용할 수 있습니다(참고문헌 [\[1\]](#references) 참조).

In [20]:
mpf_dynamic_coeffs_list = []
for t in trotter_times:
    print(f"Computing dynamic coefficients for time={t}")
    lse = setup_dynamic_lse(
        mpf_trotter_steps,
        t,
        identity_factory,
        exact_factory,
        approx_factory,
        mps_initial_state,
    )
    problem, coeffs = setup_frobenius_problem(lse)
    try:
        problem.solve()
        mpf_dynamic_coeffs_list.append(coeffs.value)
    except Exception as error:
        mpf_dynamic_coeffs_list.append(np.zeros(len(mpf_trotter_steps)))
        print(error, "Calculation Failed for time", t)
    print("")

Computing dynamic coefficients for time=0.5

Computing dynamic coefficients for time=0.6

Computing dynamic coefficients for time=0.7

Computing dynamic coefficients for time=0.7999999999999999

Computing dynamic coefficients for time=0.8999999999999999

Computing dynamic coefficients for time=0.9999999999999999

Computing dynamic coefficients for time=1.0999999999999999

Computing dynamic coefficients for time=1.1999999999999997

Computing dynamic coefficients for time=1.2999999999999998

Computing dynamic coefficients for time=1.4

Computing dynamic coefficients for time=1.4999999999999998



#### Combine Trotter expectation values with the MPF coefficients

Now we evaluate $\langle A \rangle_{\text{MPF}}(t) = \sum_j x_j \, \langle A \rangle_{k_j}(t)$ for each set of coefficients (static-exact, static-approximate, and dynamic), propagate the per-circuit standard errors, and plot the resulting time series against the exact-diagonalization curve.

In [21]:
sym = {1: "^", 2: "s", 4: "p"}
# Get expectation values at all times for each Trotter step
for k, step in enumerate(mpf_trotter_steps):
    trotter_curve, trotter_curve_error = [], []
    for trotter_expvals, trotter_stds in zip(
        mpf_expvals_all_times, mpf_stds_all_times
    ):
        trotter_curve.append(trotter_expvals[k])
        trotter_curve_error.append(trotter_stds[k])

    plt.errorbar(
        trotter_times,
        trotter_curve,
        yerr=trotter_curve_error,
        alpha=0.5,
        markersize=4,
        marker=sym[step],
        color="grey",
        label=f"{mpf_trotter_steps[k]} Trotter steps",
    )

# Get expectation values at all times for the static MPF with exact coeffs
exact_mpf_curve, exact_mpf_curve_error = [], []
for trotter_expvals, trotter_stds in zip(
    mpf_expvals_all_times, mpf_stds_all_times
):
    mpf_std = np.sqrt(
        sum(
            [
                (coeff**2) * (std**2)
                for coeff, std in zip(coeffs_exact.value, trotter_stds)
            ]
        )
    )
    exact_mpf_curve_error.append(mpf_std)
    exact_mpf_curve.append(trotter_expvals @ coeffs_exact.value)

plt.errorbar(
    trotter_times,
    exact_mpf_curve,
    yerr=exact_mpf_curve_error,
    markersize=4,
    marker="o",
    label="Static MPF - Exact",
    color="purple",
)


# Get expectation values at all times for the static MPF with approximate coeffs
approx_mpf_curve, approx_mpf_curve_error = [], []
for trotter_expvals, trotter_stds in zip(
    mpf_expvals_all_times, mpf_stds_all_times
):
    mpf_std = np.sqrt(
        sum(
            [
                (coeff**2) * (std**2)
                for coeff, std in zip(coeffs_approx.value, trotter_stds)
            ]
        )
    )
    approx_mpf_curve_error.append(mpf_std)
    approx_mpf_curve.append(trotter_expvals @ coeffs_approx.value)

plt.errorbar(
    trotter_times,
    approx_mpf_curve,
    yerr=approx_mpf_curve_error,
    markersize=4,
    marker="o",
    label="Static MPF - Approx",
    color="orange",
)


# Get expectation values at all times for the dynamic MPF
dynamic_mpf_curve, dynamic_mpf_curve_error = [], []
for trotter_expvals, trotter_stds, dynamic_coeffs in zip(
    mpf_expvals_all_times, mpf_stds_all_times, mpf_dynamic_coeffs_list
):
    mpf_std = np.sqrt(
        sum(
            [
                (coeff**2) * (std**2)
                for coeff, std in zip(dynamic_coeffs, trotter_stds)
            ]
        )
    )
    dynamic_mpf_curve_error.append(mpf_std)
    dynamic_mpf_curve.append(trotter_expvals @ dynamic_coeffs)

plt.errorbar(
    trotter_times,
    dynamic_mpf_curve,
    yerr=dynamic_mpf_curve_error,
    markersize=4,
    marker="o",
    label="Dynamic MPF",
    color="pink",
)


# Exact expectation values
plt.plot(
    exact_evolution_times,
    exact_expvals,
    color="red",
    linestyle="--",
    label="Exact time-evolution",
)

plt.title(f"$\\langle Z_{{{L//2-1}}} Z_{{{L//2}}} \\rangle$ vs time")
plt.xlabel("Time")
plt.ylabel("Expectation Value")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
plt.grid(alpha=0.1)
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/multi-product-formula/extracted-outputs/35042576-0.avif" alt="Output of the previous code cell" />

##### 근사 모델을 이용한 $x$ 최적화
선택한 $k_j$ 값 집합에 대한 L1 노름이 너무 높다고 판단될 수 있습니다.
그런 경우이면서 다른 $k_j$ 값 집합을 선택할 수 없다면, 정확한 해 대신 근사 해를 사용할 수 있습니다.

이를 위해 [setup_approximate_model](https://qiskit.github.io/qiskit-addon-mpf/stubs/qiskit_addon_mpf.static.setup_approximate_model.html)을 사용하여 L1-노름을 선택한 임계값으로 제한하면서 $Ax$와 $b$의 차이를 최소화하는 다른 [cvxpy.Problem](https://www.cvxpy.org/api_reference/cvxpy.problems.html#cvxpy.Problem) 인스턴스를 구성합니다.

In [22]:
# -------------------------Step 1-------------------------
L = 50
coupling_map = CouplingMap.from_line(L, bidirectional=False)

# XXZ Hamiltonian with random couplings (Ref. [3])
np.random.seed(0)
even_edges = list(coupling_map.get_edges())[::2]
odd_edges = list(coupling_map.get_edges())[1::2]

Js = np.random.uniform(0.5, 1.5, size=L)
hamiltonian = SparsePauliOp(Pauli("I" * L))
for i, edge in enumerate(even_edges + odd_edges):
    hamiltonian += SparsePauliOp.from_sparse_list(
        [
            ("XX", (edge), 2 * Js[i]),
            ("YY", (edge), 2 * Js[i]),
            ("ZZ", (edge), 4 * Js[i]),
        ],
        num_qubits=L,
    )

observable = SparsePauliOp.from_sparse_list(
    [("ZZ", (L // 2 - 1, L // 2), 1.0)], num_qubits=L
)

total_time = 3
mpf_trotter_steps = [3, 4, 6]
order = 2
symmetric = True

# Static coefficients
lse = setup_static_lse(mpf_trotter_steps, order=order, symmetric=symmetric)
mpf_coeffs = lse.solve()
print(f"Static coefficients: {mpf_coeffs}")
print(f"L1 norm: {np.linalg.norm(mpf_coeffs, ord=1)}")

model_approx, coeffs_approx = setup_sum_of_squares_problem(
    lse, max_l1_norm=2.0
)
model_approx.solve()
print(f"Approximate coefficients: {coeffs_approx.value}")
print(f"L1 norm (approx): {np.linalg.norm(coeffs_approx.value, ord=1)}")

# -------------------------Dynamic coefficients-------------------------
single_2nd_order_circ = generate_time_evolution_circuit(
    hamiltonian, time=1.0, synthesis=SuzukiTrotter(reps=1, order=order)
)
single_2nd_order_circ = pm.run(single_2nd_order_circ)

layers = slice_by_depth(single_2nd_order_circ, max_slice_depth=1)
models = [
    LayerModel.from_quantum_circuit(layer, conserve="Sz") for layer in layers
]

approx_factory = partial(
    LayerwiseEvolver,
    layers=models,
    options={
        "preserve_norm": False,
        "trunc_params": {"chi_max": 64, "svd_min": 1e-8, "trunc_cut": None},
        "max_delta_t": 4,
    },
)

single_4th_order_circ = generate_time_evolution_circuit(
    hamiltonian, time=1.0, synthesis=SuzukiTrotter(reps=1, order=4)
)
single_4th_order_circ = pm.run(single_4th_order_circ)
exact_model_layers = [
    LayerModel.from_quantum_circuit(layer, conserve="Sz")
    for layer in slice_by_depth(single_4th_order_circ, max_slice_depth=1)
]

exact_factory = partial(
    LayerwiseEvolver,
    layers=exact_model_layers,
    dt=0.1,
    options={
        "preserve_norm": False,
        "trunc_params": {"chi_max": 64, "svd_min": 1e-8, "trunc_cut": None},
        "max_delta_t": 3,
    },
)


def identity_factory():
    return MPOState.initialize_from_lattice(models[0].lat, conserve=True)


mps_initial_state = MPS_neel_state(models[0].lat)

print(f"Computing dynamic coefficients for time={total_time}")
lse_dyn = setup_dynamic_lse(
    mpf_trotter_steps,
    total_time,
    identity_factory,
    exact_factory,
    approx_factory,
    mps_initial_state,
)
problem, coeffs_dyn = setup_frobenius_problem(lse_dyn)
try:
    problem.solve()
    mpf_dynamic_coeffs = coeffs_dyn.value
except Exception as error:
    mpf_dynamic_coeffs = np.zeros(len(mpf_trotter_steps))
    print(error, "Calculation Failed")

# -------------------------Step 1 (cont): Build circuits-------------------------
mpf_circuits = []
for k in mpf_trotter_steps:
    circuit = QuantumCircuit(L)
    circuit.x([i for i in range(L) if i % 2])
    trotter_circ = generate_time_evolution_circuit(
        hamiltonian,
        synthesis=SuzukiTrotter(reps=k, order=order),
        time=total_time,
    )
    circuit.compose(trotter_circ, qubits=range(L), inplace=True)
    mpf_circuits.append(circuit)

# Baseline "single deep circuit" comparison run with k=10 Trotter steps.
# Its two-qubit depth is deeper than the deepest MPF constituent (k_max=6) plus
# the overhead of running multiple circuits, pushing it into the noise-limited
# regime where MPF is expected to outperform. It does NOT target the MPF's effective
# Trotter error (which would require many more steps).
comp_circuit = QuantumCircuit(L)
comp_circuit.x([i for i in range(L) if i % 2])
trotter_circ = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=10, order=order),
    time=total_time,
)
comp_circuit.compose(trotter_circ, qubits=range(L), inplace=True)
mpf_circuits.append(comp_circuit)

Static coefficients: [ 0.42857143 -1.82857143  2.4       ]
L1 norm: 4.65714285714286
Approximate coefficients: [-0.4942491   0.40206845  1.09218065]
L1 norm (approx): 1.9884981979026675
Computing dynamic coefficients for time=3


We now optimize the circuits for the chosen backend. We use the Qiskit preset pass manager at `optimization_level=3`, which automatically selects a good set of physical qubits and routes each circuit onto the device topology.

In [23]:
# -------------------------Step 2-------------------------
service = QiskitRuntimeService()
# backend = service.least_busy(operational=True, simulator=False, min_num_qubits=L)
backend = service.backend("ibm_fez")
print(backend)

transpiler = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)
transpiled_circuits = [transpiler.run(circ) for circ in mpf_circuits]

isa_observables = [
    observable.apply_layout(circ.layout) for circ in transpiled_circuits
]

<IBMBackend('ibm_fez')>


이 최적화 문제를 해결하는 방법에는 완전한 자유가 있으므로, 최적화 솔버, 수렴 임계값 등을 변경할 수 있습니다.
[근사 모델 사용 방법 가이드](https://qiskit.github.io/qiskit-addon-mpf/how_tos/using_approximate_model.html)를 확인하세요.
#### Dynamic MPF coefficients
이전 섹션에서는 표준 Trotter 근사를 개선하는 정적 MPF를 소개했습니다. 그러나 이 정적 버전이 반드시 근사 오류를 최소화하는 것은 아닙니다. 구체적으로, $\mu^S(t)$로 표기되는 정적 MPF는 곱 공식 상태 ${\rho_{k_i}(t)}_{i=1}^r$로 생성되는 부분 공간으로의 $\rho(t)$의 최적 사영이 아닐 수 있습니다.

이를 해결하기 위해, Frobenius 노름 의미에서 근사 오류를 최소화하는 동적 MPF(참고문헌 [\[2\]](#references)에서 도입되었고 참고문헌 [\[3\]](#references)에서 실험적으로 검증됨)를 고려합니다. 형식적으로, 다음을 최소화하는 데 집중합니다:

$$
\|\rho(t) - \mu^D(t)\|_F^2 \;=\; \mathrm{Tr}\bigl[ \left( \rho(t) - \mu^D(t)\right)^2 \bigr],
$$

각 시간 $t$에서의 계수 $x_i(t)$에 대해. Frobenius 노름에서의 *최적* 사영자는 $\mu^D(t) \;=\; \sum_{i=1}^r x_i(t)\,\rho_{k_i}(t)$이며, 이를 *동적* MPF라고 합니다. 위의 정의를 대입하면:

$$
\|\rho(t) - \mu^D(t)\|_F^2
\;=\; \\
= \mathrm{Tr}\bigl[ \left( \rho(t) - \mu^D(t)\right)^2 \bigr]
\;=\; \\
= \mathrm{Tr}\bigl[ \left( \rho(t) - \sum_{i=1}^r x_i(t)\,\rho_{k_i}(t) \right) \left(  \rho(t) - \sum_{j=1}^r x_j(t)\,\rho_{k_j}(t) \right) \bigr]
\;=\; \\
= 1 \;+\; \sum_{i,j=1}^r M_{i,j}(t)\,x_i(t)\,x_j(t)
\;-\;
2 \sum_{i=1}^r L_i^{\mathrm{exact}}(t)\,x_i(t),
$$

여기서 $M_{i,j}(t)$는 *그람 행렬(Gram matrix)*로, 다음과 같이 정의됩니다:

$$
M_{i,j}(t) \;=\; \mathrm{Tr}\bigl[\rho_{k_i}(t)\,\rho_{k_j}(t)\bigr]
\;=\;
\bigl|\langle \psi_{\mathrm{in}} \!\mid S\bigl(t/k_i\bigr)^{-k_i}\,S\bigl(t/k_j\bigr)^{k_j} \!\mid \psi_{\mathrm{in}} \rangle \bigr|^2.
$$

그리고

$$
L_i^{\mathrm{exact}}(t) = \mathrm{Tr}[\rho(t)\,\rho_{k_i}(t)]
$$

는 정확한 상태 $\rho(t)$와 각 곱 공식 근사 $\rho_{k_i}(t)$ 사이의 중첩(overlap)을 나타냅니다. 실제 상황에서 이 중첩들은 노이즈나 $\rho(t)$에 대한 부분적 접근으로 인해 근사적으로만 측정될 수 있습니다.

여기서 $\lvert\psi_{\mathrm{in}}\rangle$은 초기 상태이고, $S(\cdot)$는 곱 공식에서 적용되는 연산입니다. 이 식을 최소화하는 계수 $x_i(t)$를 선택하고 ($\rho(t)$가 완전히 알려지지 않은 경우 근사적인 중첩 데이터를 처리하여), MPF 부분 공간 내에서 $\rho(t)$의 (Frobenius 노름 의미의) "최상의" 동적 근사를 얻습니다. $L_i(t)$와 $M_{i,j}(t)$의 양은 텐서 네트워크 방법으로 효율적으로 계산할 수 있습니다 [\[3\]](#references). MPF Qiskit 애드온은 이 계산을 수행하기 위한 여러 "백엔드"를 제공합니다. 아래 예시는 가장 유연한 방법을 보여주며, [TeNPy 레이어 기반 백엔드](https://qiskit.github.io/qiskit-addon-mpf/apidocs/qiskit_addon_mpf.backends.tenpy_layers.html#module-qiskit_addon_mpf.backends.tenpy_layers) 문서에도 자세히 설명되어 있습니다. 이 방법을 사용하려면, 원하는 시간 진화를 구현하는 Circuit에서 시작하여 해당 Circuit의 레이어에서 이러한 연산을 나타내는 모델을 생성합니다. 마지막으로, 시간 진화된 양 $M_{i,j}(t)$와 $L_i(t)$를 생성하는 데 사용할 수 있는 `Evolver` 객체를 생성합니다. 먼저 Circuit으로 구현된 근사 시간 진화([`ApproxEvolverFactory`](https://qiskit.github.io/qiskit-addon-mpf/apidocs/qiskit_addon_mpf.dynamic.html#qiskit_addon_mpf.dynamic.ApproxEvolverFactory))에 해당하는 `Evolver` 객체를 생성합니다. 특히, `order` 변수가 일치하는지 각별히 주의하세요. 근사 시간 진화에 해당하는 Circuit을 생성할 때, `time = 1.0` 및 Trotter 단계 수(`reps=1`)에 대한 플레이스홀더 값을 사용합니다. 올바른 근사 Circuit은 이후 `setup_dynamic_lse`의 동적 문제 해결자에 의해 생성됩니다.

In [24]:
# -------------------------Step 3-------------------------
estimator = Estimator(mode=backend)
estimator.options.default_shots = 30000

# Error suppression/mitigation
estimator.options.dynamical_decoupling.enable = True
estimator.options.twirling.enable_gates = True
estimator.options.twirling.enable_measure = True
estimator.options.twirling.num_randomizations = "auto"
estimator.options.twirling.strategy = "active-accum"
estimator.options.resilience.measure_mitigation = True
estimator.options.experimental.execution_path = "gen3-turbo"

estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 1.2, 1.4)
estimator.options.resilience.zne.extrapolator = "linear"

estimator.options.environment.job_tags = ["TUT_MPF"]

job_50 = estimator.run(
    [
        (circ, observable)
        for circ, observable in zip(transpiled_circuits, isa_observables)
    ]
)

> **Warning:** 텐서 네트워크 시뮬레이션의 세부 사항을 결정하는 `LayerwiseEvolver`의 옵션은 잘못 정의된 최적화 문제를 설정하지 않도록 신중하게 선택해야 합니다.
그런 다음 정확한 진화자(예: [`ExactEvolverFactory`](https://qiskit.github.io/qiskit-addon-mpf/apidocs/qiskit_addon_mpf.dynamic.html#qiskit_addon_mpf.dynamic.ExactEvolverFactory))를 설정합니다. 이는 실제 또는 "참조" 시간 진화를 계산하는 [`Evolver`](https://qiskit.github.io/qiskit-addon-mpf/apidocs/qiskit_addon_mpf.backends.html#qiskit_addon_mpf.backends.Evolver) 객체를 반환합니다. 현실적으로는, 더 높은 차수의 Suzuki–Trotter 공식 또는 작은 시간 단계를 사용하는 다른 신뢰할 수 있는 방법으로 정확한 진화를 근사할 것입니다. 아래에서는 작은 시간 단계 `dt=0.1`을 사용하는 4차 Suzuki-Trotter 공식으로 정확한 시간 진화 상태를 근사하는데, 이는 시간 $t$에서 사용되는 Trotter 단계 수가 $k=t/dt$임을 의미합니다. 또한 기저 텐서 네트워크의 최대 본드 차원과 분할된 텐서 네트워크 본드의 최소 특이값을 제한하기 위해 일부 TeNPy 전용 절단 옵션을 지정합니다. 이 매개변수들은 동적 MPF 계수로 계산된 기댓값의 정확도에 영향을 미칠 수 있으므로, 계산 시간과 정확도 사이의 최적 균형을 찾기 위해 다양한 값 범위를 탐색하는 것이 중요합니다. MPF 계수의 계산은 하드웨어 실행으로 얻은 PF의 기댓값에 의존하지 않으므로, 후처리에서 조정할 수 있습니다.

In [25]:
# -------------------------Step 4-------------------------
result = job_50.result()
evs = [res.data.evs for res in result]
std = [res.data.stds for res in result]

print(evs)
print(std)

[array(-0.07916195), array(-0.04479681), array(-0.2560756), array(-0.06045848)]
[array(0.04605538), array(0.10056336), array(0.14426151), array(0.04059092)]


In [26]:
exact_mpf_std = np.sqrt(
    sum([(coeff**2) * (std**2) for coeff, std in zip(mpf_coeffs, std[:3])])
)
print(
    "Exact static MPF expectation value: ",
    evs[:3] @ mpf_coeffs,
    "+-",
    exact_mpf_std,
)
approx_mpf_std = np.sqrt(
    sum(
        [
            (coeff**2) * (std**2)
            for coeff, std in zip(coeffs_approx.value, std[:3])
        ]
    )
)
print(
    "Approximate static MPF expectation value: ",
    evs[:3] @ coeffs_approx.value,
    "+-",
    approx_mpf_std,
)
dynamic_mpf_std = np.sqrt(
    sum(
        [
            (coeff**2) * (std**2)
            for coeff, std in zip(mpf_dynamic_coeffs, std[:3])
        ]
    )
)
print(
    "Dynamic MPF expectation value: ",
    evs[:3] @ mpf_dynamic_coeffs,
    "+-",
    dynamic_mpf_std,
)

Exact static MPF expectation value:  -0.5665938395816946 +- 0.3925273058119915
Approximate static MPF expectation value:  -0.25856647611537903 +- 0.164249927266166
Dynamic MPF expectation value:  -0.12667812062949296 +- 0.06059471006973169


In [27]:
sym = {3: "^", 4: "s", 6: "p"}
for k, step in enumerate(mpf_trotter_steps):
    plt.errorbar(
        k,
        evs[k],
        yerr=std[k],
        alpha=0.5,
        markersize=4,
        marker=sym[step],
        color="grey",
        label=f"{mpf_trotter_steps[k]} Trotter steps",
    )

plt.errorbar(
    3,
    evs[-1],
    yerr=std[-1],
    alpha=0.5,
    markersize=8,
    marker="x",
    color="blue",
    label="10 Trotter steps",
)

plt.errorbar(
    4,
    evs[:3] @ mpf_coeffs,
    yerr=exact_mpf_std,
    markersize=4,
    marker="o",
    color="purple",
    label="Static MPF",
)

plt.errorbar(
    5,
    evs[:3] @ coeffs_approx.value,
    yerr=approx_mpf_std,
    markersize=4,
    marker="o",
    color="orange",
    label="Approximate static MPF",
)

plt.errorbar(
    6,
    evs[:3] @ mpf_dynamic_coeffs,
    yerr=dynamic_mpf_std,
    markersize=4,
    marker="o",
    color="pink",
    label="Dynamic MPF",
)

exact_obs = -0.24384471447172074  # Calculated via Tensor Network calculation
plt.axhline(
    y=exact_obs, linestyle="--", color="red", label="Exact time-evolution"
)

plt.title(
    f"$\\langle Z_{{{L//2-1}}} Z_{{{L//2}}} \\rangle$ at time {total_time} for the different methods"
)
plt.xlabel("Method")
plt.ylabel("Expectation Value")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
plt.grid(alpha=0.1)
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/multi-product-formula/extracted-outputs/64360d85-0.avif" alt="Output of the previous code cell" />

A few observations about the hardware results above:

- **Going deeper is not free on hardware.** The single-circuit baselines tell the story directly: the $k = 6$ circuit is essentially exact ($-0.256$ versus the reference $-0.244$), yet the deeper $k = 10$ baseline is *worse* ($-0.061$, off by $\sim 0.18$), not better. Once the Trotter error is already small, adding steps mostly deepens the circuit and accumulates more gate noise and decoherence. This is precisely the regime MPFs are built for: reach the accuracy of a deep circuit using only shallow constituents.

- **A small-norm MPF beats the deep single circuit.** The approximate-static MPF (capped at $\|x\|_1 \approx 2$) lands at $-0.259$, within $\sim 0.015$ of the reference and far closer than the $k = 10$ baseline. The dynamic MPF ($-0.127$) also comfortably beats that baseline. Both combine only the shallow $k_j = [3, 4, 6]$ circuits, yet recover an answer the deep single circuit could not.

- **Coefficient norm matters more than mathematical optimality.** The exact-static MPF has $\|x\|_1 = 4.66$ and is the *worst* estimator of all ($-0.567$, off by more than $0.3$): the large coefficient norm amplifies the residual gate noise, decoherence, and ZNE error on each $\langle A \rangle_{k_j}$ by roughly the same factor, overwhelming the Trotter-error cancellation it buys. Capping the norm (the approximate-static solver, $\|x\|_1 \approx 2$) removes this overwhelm and gives the best estimate &mdash; even though its coefficients no longer cancel the leading Trotter error exactly.

- **Individual shallow circuits can still be competitive.** The lone $k = 6$ constituent ($-0.256$) is itself essentially exact here &mdash; on this run it is even marginally closer than the approximate-static MPF. The catch is that you do not know in advance *which* single $k$ sits in the sweet spot of "converged but not yet noise-limited," and the safe-looking choice of simply going deeper ($k = 10$) to guarantee Trotter convergence is exactly the one that fails. The MPF gives a principled combination of shallow circuits that does not require guessing the right depth.

The practical takeaway is that on hardware, MPFs should be paired with strong error mitigation on each individual $\langle A \rangle_{k_j}$, the coefficient $L_1$-norm should be kept modest (use the approximate solver, or the dynamic MPF), and the Trotter steps $k_j$ should be chosen so that $t/k_{\min} \lesssim 1$ &mdash; here $k_{\min} = 3$ at $t = 3$ gives $t/k_{\min} = 1$, keeping the constituents inside the convergent regime where the leading-error model the static MPF relies on is valid. With those choices, the small-norm MPFs here match a converged single circuit while the naive "just go deeper" baseline does not, recovering the depth-versus-accuracy advantage shown in Ref. [\[3\]](#references). Note also that individual runs are noisy &mdash; on a different submission of the same job (or a different backend), the exact ordering can shift; the robust trends are that small-$\|x\|_1$ MPFs do well, the large-$\|x\|_1$ exact-static MPF is amplified by hardware noise, and the over-deep single circuit is noise-limited.

## Next steps
<Admonition type="tip" title="Recommendations">

If you found this work interesting, you might be interested in the following material:
- [How to choose the Trotter steps for an MPF](https://qiskit.github.io/qiskit-addon-mpf/how_tos/choose_trotter_steps.html) — practical guidance on selecting $k_j$ values to avoid instabilities
- [How to use the approximate model](https://qiskit.github.io/qiskit-addon-mpf/how_tos/using_approximate_model.html) — tuning the $L_1$-norm constraint and solver options for the approximate static MPF
- [`qiskit-addon-mpf` API reference](https://qiskit.github.io/qiskit-addon-mpf/) — full documentation for static, dynamic, and backend modules
</Admonition>

## References

\[1] Vazquez, A. C., Egger, D. J., Ochsner, D., & Woerner, S. Well-conditioned multi-product formulas for hardware-friendly Hamiltonian simulation. [Quantum, 7, 1067 (2023)](https://quantum-journal.org/papers/q-2023-07-25-1067/)

\[2] Zhuk, S., Robertson, N. F., & Bravyi, S. Trotter error bounds and dynamic multi-product formulas for Hamiltonian simulation. [Physical Review Research, 6(3), 033309 (2024)](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.6.033309)

\[3] Robertson, N. F., et al. Tensor network enhanced dynamic multiproduct formulas. [arXiv:2407.17405 (2024)](https://arxiv.org/abs/2407.17405)